# PCA 降维问题

### R PCA 代码

In [ ]:
#' Compute the PCA projection
#'
#' @param C data matrix used for PCA projection
#' @param L number for the top principal components
#' @import irlba irlba
#' @importFrom stats qnorm
#' @export
pca_projection_R <- function(C, L) {
    if (L >= min(dim(C))){
        eigen_res <- eigen(C)

        U <- eigen_res$vector
        V <- eigen_res$value
        eig_sort <- sort(V, decreasing = T, index.return = T)
        eig_idx <- eig_sort$ix

        W <- U[, eig_idx[1:L]]
        return (W)
    } else{
        initial_v <- as.matrix(qnorm(1:(ncol(C) + 1)/(ncol(C) + 1))[1:ncol(C)])
        eigen_res <- irlba::irlba(C, nv = L, v = initial_v)
        U <- eigen_res$u
        V <- eigen_res$v
        return (V)
    }
}


# 指定CSV文件路径
R_X_path = "random_matrix.csv"


# header = TRUE 表示文件包含列名，可根据实际情况修改
X <- as.matrix(read.csv(R_X_path, header = FALSE)) 

result <- pca_projection_R(
    X, 2
)

print(result)

### Python PCA 代码

In [ ]:
import numpy as np
import os
import sys
from scipy.linalg import eigh  # 用于进行特征分解
from sklearn.cluster import KMeans
from utils import *
from sklearn.utils.extmath import randomized_svd
import pandas as pd

def pca_projection_python(C, L):
    # C: 用于PCA的数据矩阵
    # L: 要计算的主成分的数量
    # num_features, num_samples = C.shape

    # # 判断L是否大于等于矩阵C的最小维度
    # if L >= min(num_features, num_samples):
    #     # 计算矩阵C的特征值和特征向量
    #     if num_features < num_samples:
    #         cov_matrix = np.cov(C.T)
    #     else:
    #         cov_matrix = np.cov(C)

    #     eigen_values, eigen_vectors = eigh(cov_matrix)

    #     # 按特征值降序排序，获取排序的索引
    #     sorted_indices = np.argsort(eigen_values)[::-1]

    #     # 获取前L个对应最大特征值的特征向量
    #     W = eigen_vectors[:, sorted_indices[:L]]
    #     return W
    # else:
    #     # 确保生成的初始向量具有正确的大小
    #     n_features = C.shape[1]
    #     initial_v = norm.ppf(np.linspace(1 / (n_features + 1), 1, n_features))

    #     # 使用稀疏SVD进行近似PCA，当L小于维度时
    #     u, s, vt = svds(C, k=L, v0=initial_v)

    #     vt[[0, 1]] = vt[[1, 0]]

    #     logger.warning(f"结束python pca降维")
    #     # 返回前L个右奇异向量（在R中是V）
    #     return vt.T
        # Check if number of components requested is less than the min dimension
    if L >= min(C.shape):
        # Eigenvalue decomposition
        cov_matrix = np.cov(C, rowvar=False)
        eigenvalues, eigenvectors = eigh(cov_matrix)

        # Sort eigenvalues and eigenvectors
        eig_sort_idx = np.argsort(eigenvalues)[::-1]
        eig_idx = eig_sort_idx[:L]

        # Select top L eigenvectors
        W = eigenvectors[:, eig_idx]
        return W
    else:
        # Use randomized SVD for larger datasets
        initial_v = np.quantile(np.random.rand(len(C[0])), q=np.linspace(0, 1, len(C[0])+1))[1:len(C[0])+1]
        U, S, Vt = randomized_svd(C, n_components=L, random_state=42)
        return Vt.T
    
    
# 指定CSV文件路径
R_X_path = "random_matrix.csv"

# 使用pandas读取CSV文件
df = pd.read_csv(R_X_path, header=None)  # header=None表示不将第一行作为列名
X_R = df.values  # 或使用 df.to_numpy()

result = pca_projection_python(X_R, 2)

print(result)


### 测试数据

In [ ]:
import numpy as np
import pandas as pd

def generate_random_matrix(rows, cols, filename):
    """生成一个随机矩阵并保存为CSV文件。

    参数:
        rows (int): 矩阵的行数。
        cols (int): 矩阵的列数。
        filename (str): 保存CSV文件的名称。
    """
    np.random.seed(42)
    
    # 生成一个指定大小的随机矩阵
    random_matrix = np.random.rand(rows, cols)

    # 将随机矩阵转换为DataFrame以便于存储
    df = pd.DataFrame(random_matrix)

    # 将DataFrame保存为CSV文件
    df.to_csv(filename, index=False, header=False)  # 不保存行号和列名

    print(f"随机矩阵已保存为 {filename}")

# 设置矩阵的行数和列数
num_rows = 300  # 行数
num_cols = 300   # 列数
output_filename = './random_matrix.csv'  # 输出文件名

# 调用函数生成随机矩阵并保存为CSV文件
generate_random_matrix(num_rows, num_cols, output_filename)
